In [1]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [2]:
feat = pd.read_csv('credit_card_featured.csv')

In [3]:
raw_cols = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
       'SEP_REPAY_STATUS', 'AUG_REPAY_STATUS', 'JUL_REPAY_STATUS',
       'JUN_REPAY_STATUS', 'MAY_REPAY_STATUS', 'APR_REPAY_STATUS',
       'SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT']
payment_ratio_cols = ['PAY_RATIO_SEP', 'PAY_RATIO_AUG',
       'PAY_RATIO_JUL', 'PAY_RATIO_JUN', 'PAY_RATIO_MAY']

In [4]:
X = feat
y = X['default.payment.next.month']
X = feat[raw_cols + payment_ratio_cols]

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [6]:
logreg = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0, add_indicator=True)),
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

In [8]:
model = XGBClassifier(eval_metric = 'logloss', random_state=0)

In [9]:
probs_by_model = {}
for name, m in [('logreg', logreg), ('xgb', model)]:
    probs_by_model[name] = cross_val_predict(m, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

# xgb is the stronger model (see 02_baseline.ipynb / 04_modeling.ipynb), so it drives
# the threshold/cost analysis below; logreg's predictions are kept in probs_by_model
# for reference.
probs = probs_by_model['xgb']

In [10]:
print(probs)

[0.1188513  0.02051715 0.09856749 ... 0.1300228  0.36571065 0.40841353]


In [11]:
tn, fp, fn, tp = confusion_matrix(y_train, (probs >= 0.5).astype(int)).ravel()
print(tn, fp, fn, tp)

17538 1153 3432 1877


In [12]:
COST_FP = 50

In [13]:
def sweep(cost_fn):
    COST_FN = cost_fn
    rows = []
    for t in np.arange(0.01, 0.96, 0.01):
        tn, fp, fn, tp = confusion_matrix(y_train, (probs >= t).astype(int)).ravel()
        rows.append({
            'threshold': t,
            'flagged': tp + fp,
            'fn': fn,
            'fp': fp,
            'cost': fn * COST_FN + fp * COST_FP,
            'recall': tp / (tp + fn),
            'precision': tp / (tp + fp) if (tp + fp) > 0 else 0
        })

    df_sweep = pd.DataFrame(rows)

    best = df_sweep.loc[df_sweep['cost'].idxmin()]
    return df_sweep, best

In [14]:
table, best = sweep(1000)
print(best)


threshold         0.010000
flagged       23197.000000
fn               32.000000
fp            17920.000000
cost         928000.000000
recall            0.993972
precision         0.227486
Name: 0, dtype: float64


In [15]:
table_5to1, best_5to1 = sweep(250)
print(best_5to1)

threshold         0.150000
flagged       10491.000000
fn             1445.000000
fp             6627.000000
cost         692600.000000
recall            0.727821
precision         0.368316
Name: 14, dtype: float64


In [16]:
n_train = len(y_train)
cap_rows = []
for pct in [0.05, 0.10, 0.20]:
    target = pct * n_train
    row = table.loc[(table['flagged'] - target).abs().idxmin()]
    cap_rows.append({
        'capacity_pct': pct,
        'threshold': round(row['threshold'], 2),
        'flagged': int(row['flagged']),
        'recall': round(row['recall'], 3),
        'precision': round(row['precision'], 3),
    })

capacity_table = pd.DataFrame(cap_rows)
print(capacity_table)

   capacity_pct  threshold  flagged  recall  precision
0          0.05       0.77     1225   0.168      0.727
1          0.10       0.58     2435   0.304      0.662
2          0.20       0.34     4891   0.486      0.528


Under the assumed cost ratio of 20:1, the cost-minimizing threshold falls below 0.01, flagging nearly the entire portfolio. This is a property of the assumption, not a useful policy: when a miss is 20x a false alarm and 22% of customers default, blanket intervention beats any selective strategy. Only at a 5:1 ratio does a selective threshold (0.15) emerge.

Contacting the model's top-ranked 10% of customers identifies 30% of all defaulters — three times what random selection would achieve — with two-thirds of contacts reaching a customer who does go on to default.